 # 05_evaluate.py : collect LOOCV results of C0, C0+, C0n, C1, C2, C3 and report

 Average F1 (all classes / without Other / target classes), per-class F1 (mean and SD over folds), confusion matrices

 (stitched test recordings, pooled over folds and seeds) and per-class gain over C0+. Ungated (final round) is primary; gated is also reported.

 Does not load any recording data: it only reads the unit files written by 01-04.

In [ ]:
from thesis_common import *      # constants and plotting helpers only; no data is loaded here
from sklearn.metrics import confusion_matrix

EVAL = OUT / "eval"; EVAL.mkdir(exist_ok=True)

rows = []
for stage in ("C0", "C1", "C2", "C3"):
    for p in sorted((OUT / f"units_{stage}").glob("*.pkl")): rows += pickle.load(open(p, "rb"))["rows"]
RES = pd.DataFrame(rows); RES.drop(columns="cm").to_csv(EVAL / "all_results.csv", index=False)
print(RES.groupby(["cond", "kind"]).size())
print("folds per condition:", RES.groupby("cond").fold.nunique().to_dict())


In [ ]:
def per_class_table(gated=False):
    d = RES[RES.frac == 1.0]; sets = {"C0": d[d.kind == "baseline"], "C0+": d[(d.kind == "ctrl") & (d["round"] == FINAL)], "C0n": d[d.kind == "noise"],
        "C1": d[(d.cond == "C1") & (d.kind == ("gated" if gated else "pseudo")) & (d.pool == "A") & (d.pool_frac == 1.0) & ((d["round"] == FINAL) | gated)],
        "C2": d[(d.cond == "C2") & (d.kind == ("gated" if gated else "pseudo")) & (d.pool == "A") & (d.pool_frac == 1.0) & ((d["round"] == FINAL) | gated)],
        "C3": d[(d.kind == "c3") & (d.pool == "A") & (d.pool_frac == 1.0)]}
    return {k: v for k, v in sets.items() if len(v)}

def boot_ci(d, n=10000, seed=0):
    rng = np.random.RandomState(seed); return np.percentile(d[rng.randint(0, len(d), (n, len(d)))].mean(1), [2.5, 97.5])


 ## Average F1 (stitched test recordings; seeds averaged within fold, then mean and SD over folds)

In [ ]:
def avg_f1_table(gated=False):
    out = {}
    for k, v in per_class_table(gated).items():
        f = v.groupby("fold")[["test_macro", "test_macroNoOther", "test_target"]].mean()
        out[k] = {"n_folds": len(f), "avg F1": f.test_macro.mean(), "sd": f.test_macro.std(), "avg F1 (no Other)": f.test_macroNoOther.mean(),
                  "sd (no Other)": f.test_macroNoOther.std(), "target F1": f.test_target.mean(), "sd (target)": f.test_target.std()}
    return pd.DataFrame(out).T

for gated in (False, True):
    t = avg_f1_table(gated); print(f"\n######## Average F1 | {'GATED' if gated else 'UNGATED (final round), primary'} | pool A ########")
    display(t.round(4)); t.to_csv(EVAL / f"avg_F1_{'gated' if gated else 'ungated'}.csv")


 ## Per-class F1 and confusion matrices

In [ ]:
sets = per_class_table()
pc = pd.DataFrame({k: v.groupby("fold")[[f"test_F1_{n}" for n in CLASS_NAMES]].mean().mean() for k, v in sets.items()}); pc.index = CLASS_NAMES
pcs = pd.DataFrame({k: v.groupby("fold")[[f"test_F1_{n}" for n in CLASS_NAMES]].mean().std() for k, v in sets.items()}); pcs.index = CLASS_NAMES
print("Mean per-class test F1 over folds"); display(pc.round(3)); print("SD over folds"); display(pcs.round(3))
pc.to_csv(EVAL / "per_class_F1_mean.csv"); pcs.to_csv(EVAL / "per_class_F1_sd.csv")
pc.loc[TARGETS + ["Resting", "Swimming", "Breathing"]].plot.bar(figsize=(9, 4), ylabel="mean per-class test F1")
plt.savefig(OUT / "fig" / "per_class_F1.png", dpi=200, bbox_inches="tight"); plt.show()

fig, axs = plt.subplots(1, len(sets), figsize=(4.2 * len(sets), 4))
for ax, (k, v) in zip(np.atleast_1d(axs), sets.items()):
    cm = np.sum([c for c in v["cm"]], axis=0).astype(float); pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(EVAL / f"confusion_counts_{k}.csv")
    cm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    ax.imshow(cm, vmin=0, vmax=1, cmap="Blues"); ax.set_title(k); ax.set_xticks(range(NC)); ax.set_yticks(range(NC))
    ax.set_xticklabels([n[:4] for n in CLASS_NAMES], rotation=60); ax.set_yticklabels([n[:4] for n in CLASS_NAMES])
    for i in range(NC):
        for j in range(NC): ax.text(j, i, f"{cm[i,j]:.2f}", ha="center", va="center", fontsize=6, color="w" if cm[i, j] > .5 else "k")
fig.suptitle("Row-normalised confusion (stitched test recordings, pooled over folds and seeds): rows = true, cols = predicted")
fig.savefig(OUT / "fig" / "confusion_matrices.png", dpi=200, bbox_inches="tight"); plt.show()


 ## Per-class gain over C0+

In [ ]:
COL = {"C1": "tab:blue", "C2": "tab:orange", "C3": "tab:green"}

def plot_per_class_gain():
    """Per-class test F1 gain over the C0+ control (mean over folds, 95% bootstrap CI of paired fold differences)."""
    cols = [f"test_F1_{n}" for n in CLASS_NAMES]; sets = per_class_table()
    if "C0+" not in sets: print("no C0+ rows yet"); return
    fl = {k: v.groupby("fold")[cols].mean() for k, v in sets.items()}; conds = [c for c in ("C1", "C2", "C3") if c in fl]
    fig, ax = plt.subplots(figsize=(10, 4.6)); wd = .8 / max(1, len(conds)); x = np.arange(NC)
    for i, c in enumerate(conds):
        diff = (fl[c] - fl["C0+"]).reindex(columns=cols); mu, lo, hi = [], [], []
        for cl in cols:
            d = diff[cl].dropna().values; mu.append(d.mean() if len(d) else np.nan)
            l, h = boot_ci(d) if len(d) >= 3 else (np.nan, np.nan); lo.append(l); hi.append(h)
        mu, lo, hi = map(np.array, (mu, lo, hi)); pos = x + (i - (len(conds) - 1) / 2) * wd
        ax.bar(pos, np.nan_to_num(mu), wd, color=COL[c], label=c, alpha=.9)
        ax.errorbar(pos, np.nan_to_num(mu), yerr=[np.nan_to_num(mu - lo), np.nan_to_num(hi - mu)], fmt="none", ecolor="k", lw=1, capsize=2)
    ax.axhline(0, c="k", lw=.8); ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES); ax.set_ylabel("Δ per-class test F1 vs C0+")
    for lab in ax.get_xticklabels():
        if lab.get_text() in TARGETS: lab.set_fontweight("bold")
    ax.set_title("Where do the gains come from? (bold = target classes; pool A, final round)"); ax.legend()
    fig.tight_layout(); fig.savefig(OUT / "fig" / "per_class_gain_vs_C0plus.png", dpi=200, bbox_inches="tight"); plt.show()

plot_per_class_gain()
print("All tables/figures saved in", EVAL, "and", OUT / "fig")
